# DFU Repair-7 CPU WORKING
Small Colab-safe loader. CPU only. Preserves the 38 good trials and authorizes only the exact seven fold-1 repairs.

In [ ]:
import ast, json, urllib.request

V17_URL = "https://raw.githubusercontent.com/AzizulHakim00/DFU-ImageGuard/ba99c6b6da8ebd5a771e0eef6424586ee0933c02/notebooks/DFU_Repair7_v1_7_CPU_Colab.ipynb"

raw = urllib.request.urlopen(V17_URL, timeout=120).read()
nb = json.loads(raw.decode("utf-8"))
cells = [c for c in nb.get("cells", []) if c.get("cell_type") == "code"]
if len(cells) != 1:
    raise RuntimeError(f"Expected exactly one v1.7 code cell, found {len(cells)}")
code = "".join(cells[0]["source"])

old = '''# Static safety assertions before execution.
if script.count("rr.train_trial(") != 1:
    raise RuntimeError(f"Safety check: expected exactly one rr.train_trial call, found {script.count('rr.train_trial(')}")
for forbidden in ["rr.make_outer_folds(", "rr.assign_duplicate_groups(", "rr.build_manifest("]:
    if forbidden in script:
        raise RuntimeError(f"Safety check: forbidden split-regeneration call found: {forbidden}")
'''
new = '''# AST-based safety assertions: inspect REAL rr.* calls only.
_tree = ast.parse(script)
_rr_calls = []
for _node in ast.walk(_tree):
    if (
        isinstance(_node, ast.Call)
        and isinstance(_node.func, ast.Attribute)
        and isinstance(_node.func.value, ast.Name)
        and _node.func.value.id == "rr"
    ):
        _rr_calls.append(_node.func.attr)

if _rr_calls.count("train_trial") != 1:
    raise RuntimeError(
        f"AST safety check: expected exactly one real rr.train_trial() call, found {_rr_calls.count('train_trial')}"
    )

_forbidden = {"make_outer_folds", "assign_duplicate_groups", "build_manifest"}
_bad = sorted(_forbidden.intersection(_rr_calls))
if _bad:
    raise RuntimeError(f"AST safety check: forbidden real rr.* call(s) found: {_bad}")

print("AST safety check: PASS")
'''

if code.count(old) != 1:
    raise RuntimeError(f"Patch target expected once, found {code.count(old)}")

code = code.replace(old, new, 1)
code = code.replace("DFU Repair-7 v1.7 CPU ONLY", "DFU Repair-7 CPU WORKING", 1)
code = code.replace(
    "Pinned Repair-7 v1.7 CPU patch verification: PASS",
    "Pinned Repair-7 CPU WORKING verification: PASS",
    1,
)

compile(code, "DFU_Repair7_CPU_WORKING.py", "exec")

print("DFU Repair-7 CPU WORKING loader: PASS")
print("Notebook shell: standard nbformat")
print("CPU mode: ON")
print("38 good trials: READ-ONLY")
print("Authorized retraining: EXACT 7")
exec(compile(code, "DFU_Repair7_CPU_WORKING.py", "exec"), globals())
